# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hafsaShaban/flyrank_internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists('/content/flyrank_internship'):
    !git clone https://github.com/hafsaShaban/flyrank_internship.git
%cd /content/flyrank_internship

import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print("Loaded:", len(df), "rows")

Cloning into 'flyrank_internship'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 175 (delta 74), reused 94 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 2.28 MiB | 14.99 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/flyrank_internship
Loaded: 30000 rows


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
for col in ['impressions_90d', 'clicks_90d', 'word_count', 'ctr']:
    print(f"\n{col}:")
    print(df[col].describe())
    print("Skew:", df[col].skew())



impressions_90d:
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64
Skew: 11.384919352305715

clicks_90d:
count    30000.000000
mean        16.097333
std         75.076958
min          0.000000
25%          0.000000
50%          1.000000
75%          7.000000
max       4178.000000
Name: clicks_90d, dtype: float64
Skew: 18.345790257096372

word_count:
count    22301.000000
mean      3107.760325
std       1452.382598
min          8.000000
25%       2413.000000
50%       2877.000000
75%       3666.000000
max       9546.000000
Name: word_count, dtype: float64
Skew: 0.9379345291677674

ctr:
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64
Skew: 17.444251509644893


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Test 1: "Longer pages get more traffic."
Test 2: "Fresher (recently updated) pages have better CTR."
Test 3: "Pages with higher search volume have better average position.

In [3]:
df['log_impressions'] = np.log1p(df['impressions_90d'])

wc_tiers = df.groupby('word_count_tier').agg(
    median_log_impressions=('log_impressions','median'), n=('content_id','count'))
print("TEST 1 — word_count_tier vs impressions:")
print(wc_tiers)

fresh_tiers = df.groupby('freshness_tier').agg(
    median_ctr=('ctr','median'), n=('content_id','count'))
print("\nTEST 2 — freshness_tier vs CTR:")
print(fresh_tiers)

df['volume_bucket'] = pd.qcut(df['search_volume'], q=4, duplicates='drop')
vol_tiers = df[df['avg_position'] > 0].groupby('volume_bucket').agg(
    median_position=('avg_position','median'), n=('content_id','count'))
print("\nTEST 3 — search_volume bucket vs avg_position:")
print(vol_tiers)

TEST 1 — word_count_tier vs impressions:
                 median_log_impressions      n
word_count_tier                               
1000-2000                      5.153275   3780
2000-3500                      6.905753  11263
3500+                          7.201171   6285
<1000                          1.609438    973

TEST 2 — freshness_tier vs CTR:
                median_ctr      n
freshness_tier                   
0-30                  0.04  20480
181+                  0.00    174
31-90                 0.00    175
91-180                0.10   9171

TEST 3 — search_volume bucket vs avg_position:
                 median_position      n
volume_bucket                          
(-0.001, 10.0]              11.6  18091
(10.0, 20.0]                11.1   2245
(20.0, 74000.0]             13.1   6740


/tmp/ipykernel_4487/2794277894.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  vol_tiers = df[df['avg_position'] > 0].groupby('volume_bucket').agg(


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Testing the assumption behind the baseline's stale_visible_page reason code: does staleness (days_since_last_update >= 180) actually associate with declining performance?

In [4]:
df['is_stale'] = (df['days_since_last_update'] >= 180)
stale_test = df.groupby('is_stale').agg(
    decline_rate=('is_declining_label','mean'), n=('content_id','count'))
print(stale_test)


          decline_rate      n
is_stale                     
False         0.542480  29826
True          0.471264    174


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [5]:
print("Total pages audited:", len(df))
print("Stale pages (n):", df['is_stale'].sum())
print("Declining rate overall:", df['is_declining_label'].mean())

Total pages audited: 30000
Stale pages (n): 174
Declining rate overall: 0.5420666666666667


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.